# Case LACEDA
Pipeline de correspondência fuzzy entre endereços de consulta (tabela CIDACS) e a base normalizada CNEFE (Bahia) para atribuir setor censitário a cada endereço.

---

## Visão Geral

| Item | Valor |
| --- | --- |
| Endereços de consulta | 107 registros |
| Registros na base CNEFE (BA) | 4.624.261 registros (após `distinct`) |
| Municípios distintos na consulta | 78 |
| Municípios distintos na CNEFE | 417 |
| Candidatos após join + filtro | 8.396.951 |
| Setores encontrados (best match) | 107 |
| Setores visualizados no mapa | 69 |

---

## Etapa 1 — Carga e Normalização das Bases

### 1.1 Tabela de Consulta (`consulta_df`)

- **Fonte:** `workspace.default.2026_teste_tecnico_relinke_cidacs_in`
- Carregada via `spark.table(...)` com verificação de existência (`try/except NameError`).
- Contém os endereços a serem enriquecidos com as colunas: `consulta_logradouro`, `consulta_numero`, `consulta_bairro`, `consulta_cep`, `consulta_complemento`, `consulta_municipio`.

### 1.2 Base CNEFE (`cnefe_df`)

- **Fonte:** `workspace.default.29_ba` (CNEFE da Bahia)
- Carregada e **normalizada** para um esquema de chaves padronizadas:
  - `chave_logradouro` = concatenação de `NOM_TIPO_SEGLOGR + NOM_TITULO_SEGLOGR + NOM_SEGLOGR` (via `F.concat_ws(" ", ...)`)
  - `chave_cep` = `CEP`
  - `chave_bairro` = `DSC_LOCALIDADE`
  - `chave_numero` = `NUM_ENDERECO` (cast para string)
  - `chave_municipio` = `COD_MUNICIPIO`
  - `setor_censitario_encontrado` = `COD_SETOR`
- Aplicado `.distinct()` para eliminar duplicatas e reduzir o volume de comparações.

---

## Etapa 2 — Diagnóstico de Compatibilidade de Municípios

### Problema identificado

A tabela de consulta usa códigos IBGE de **6 dígitos** (sem dígito verificador), enquanto a CNEFE usa códigos de **7 dígitos** (com dígito verificador).

- Exemplo: consulta = `290650` → CNEFE = `2906508`
- Comparação direta: **78 municípios não encontrados** (0% de match)

### Solução

Truncar o código CNEFE de 7 para 6 dígitos dividindo por 10 e convertendo para `long`:

```
(F.col("b.chave_municipio") / 10).cast("long")
```

Após essa correção: **0 municípios sem match** — cobertura de 100%.

---

## Etapa 3 — Matching Fuzzy com PySpark

### 3.1 Join por Município

Join `inner` entre `consulta_df` e `cnefe_df` usando a chave de município normalizada (6 dígitos). Isso reduz o escopo de comparação para apenas os endereços dentro do mesmo município.

### 3.2 Cálculo de Distâncias (Levenshtein)

Para cada par (consulta × CNEFE) dentro do mesmo município, calculam-se:

| Campo | Função | Descrição |
| --- | --- | --- |
| `street_distance` | `F.levenshtein(consulta_logradouro, chave_logradouro)` | Distância de edição entre logradouros |
| `bairro_distance` | `F.levenshtein(consulta_bairro, chave_bairro)` | Distância de edição entre bairros |

Usa-se `F.coalesce(..., F.lit(""))` para tratar valores nulos como string vazia.

### 3.3 Score de Matching

O `match_score` é uma soma ponderada de componentes:

| Componente | Peso | Condição |
| --- | --- | --- |
| CEP idêntico | +100 | `consulta_cep == chave_cep` |
| Logradouro similar | +50 − `levenshtein` | Quanto menor a distância, maior a pontuação |
| Bairro similar | +30 − `levenshtein` | Quanto menor a distância, maior a pontuação |
| Número idêntico | +20 | `consulta_numero == chave_numero` |

Score máximo teórico = 100 + 50 + 30 + 20 = **200 pontos**.

### 3.4 Filtro de Candidatos

Aplica-se filtros para eliminar correspondências implausíveis:

```
street_distance <= 50  AND  bairro_distance <= 20
```

Resultado: **8.396.951 candidatos** antes da seleção do melhor.

### 3.5 Seleção do Melhor Match (Window Function)

Usa-se `Window.partitionBy("consulta_logradouro", "consulta_numero", "consulta_cep")` ordenado por `match_score DESC`, e seleciona-se apenas `rank == 1` via `F.row_number()`.

Resultado final: **107 setores censitários encontrados** (1 por endereço de consulta).

---

## Etapa 4 — Download e Preparação do Shapefile

- **Fonte:** IBGE — Malha de setores censitários do Censo 2022 (Bahia)
- URL: `https://geoftp.ibge.gov.br/.../BA_setores_CD2022.zip`
- Download via `requests` com streaming e extração via `zipfile` para o volume `workspace.default.geo_data`.
- Shapefile carregado com `geopandas.read_file(...)`: **30.985 setores**.

---

## Etapa 5 — Visualização no Mapa

- Conversão dos resultados (`best_matches`) para pandas via `.toPandas()`.
- Join entre códigos de setor (CNEFE `COD_SETOR` com sufixo `P` → shapefile `CD_SETOR` sem sufixo).
- Mapa interativo com `folium`:
  - **Camada cinza:** limites dos 417 municípios (dissolve por `NM_MUN`)
  - **Camada verde:** 69 setores com match (polígonos + marcadores com detalhes)
  - Popups com logradouro, município, bairro, número de matches e score médio.
- Salvos em `/Volumes/workspace/default/geo_data/bahia_map.html`.

In [0]:
%python
# Ensure both dataframes are available
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load consulta_df if not already defined
try:
    consulta_df
except NameError:
    consulta_df = spark.table("workspace.default.`2026_teste_tecnico_relinke_cidacs_in`")
    print(f"Loaded consulta_df: {consulta_df.count()} records")

# Load cnefe_df if not already defined
try:
    cnefe_df
except NameError:
    cnefe_df = spark.table("workspace.default.`29_ba`").select(
        F.concat_ws(" ",
                    F.col("NOM_TIPO_SEGLOGR"),
                    F.col("NOM_TITULO_SEGLOGR"),
                    F.col("NOM_SEGLOGR")).alias("chave_logradouro"),
        F.col("CEP").alias("chave_cep"),
        F.col("DSC_LOCALIDADE").alias("chave_bairro"),
        F.col("NUM_ENDERECO").cast("string").alias("chave_numero"),
        F.col("COD_MUNICIPIO").alias("chave_municipio"),
        F.col("COD_SETOR").alias("setor_censitario_encontrado")
    ).distinct()
    print(f"Loaded cnefe_df: {cnefe_df.count()} records")
consulta_munios = consulta_df.select("consulta_municipio").distinct()
cnefe_munios = cnefe_df.select("chave_municipio").distinct()

print("=== Schema check ===")
print(f"consulta_municipio type: {consulta_df.schema['consulta_municipio'].dataType}")
print(f"chave_municipio type: {cnefe_df.schema['chave_municipio'].dataType}")

print("\n=== Sample values ===")
print("consulta_municipio samples:")
consulta_munios.show(10, truncate=False)
print("chave_municipio samples:")
cnefe_munios.show(10, truncate=False)

print(f"\nTotal distinct consulta_municipio: {consulta_munios.count()}")
print(f"Total distinct chave_municipio in CNEFE: {cnefe_munios.count()}")

# Procurar consulta_municipios NOT in CNEFE
missing_munios = consulta_munios.join(
    cnefe_munios,
    consulta_munios.consulta_municipio == cnefe_munios.chave_municipio,
    "left_anti"
)

missing_count = missing_munios.count()
print(f"\n=== Result ===")
print(f"consulta_municipios NOT found in CNEFE: {missing_count}")

# IBGE município tem 6-digit (consulta) já CNEFE 7-digit with check digit
print("=== Format comparison ===")
print("consulta_municipio (6-digit):", consulta_munios.orderBy("consulta_municipio").limit(3).collect())
print("chave_municipio  (7-digit):", cnefe_munios.orderBy("chave_municipio").limit(3).collect())

# Try matching: truncate 7-digit CNEFE code to 6 digits (drop check digit)
consulta_munios_6 = consulta_munios.withColumn("mun_6", F.substring(F.col("consulta_municipio").cast("string"), 1, 6).cast("long"))
cnefe_munios_6 = cnefe_munios.withColumn("mun_6", (F.col("chave_municipio") / 10).cast("long"))

missing_munios_6 = consulta_munios_6.join(
    cnefe_munios_6,
    consulta_munios_6.mun_6 == cnefe_munios_6.mun_6,
    "left_anti"
)

missing_count_6 = missing_munios_6.count()
print(f"\n=== Match on 6-digit code (drop check digit) ===")
print(f"consulta_municipios NOT found in CNEFE (6-digit match): {missing_count_6}")
if missing_count_6 > 0:
    print("\nStill missing:")
    missing_munios_6.select("consulta_municipio").show(truncate=False)
else:
    print("\nAll consulta_municipio values match CNEFE when comparing 6-digit codes!")
    print("Root cause: consulta_municipio uses 6-digit IBGE codes, chave_municipio uses 7-digit (with check digit)")

Loaded consulta_df: 107 records
Loaded cnefe_df: 4624261 records
=== Schema check ===
consulta_municipio type: LongType()
chave_municipio type: LongType()

=== Sample values ===
consulta_municipio samples:
+------------------+
|consulta_municipio|
+------------------+
|290650            |
|292710            |
|292870            |
|293135            |
|292880            |
|290323            |
|291560            |
|290720            |
|292265            |
|291080            |
+------------------+
only showing top 10 rows
chave_municipio samples:
+---------------+
|chave_municipio|
+---------------+
|2919553        |
|2918803        |
|2929701        |
|2909000        |
|2902252        |
|2924306        |
|2916856        |
|2903102        |
|2928901        |
|2907301        |
+---------------+
only showing top 10 rows

Total distinct consulta_municipio: 78
Total distinct chave_municipio in CNEFE: 417

=== Result ===
consulta_municipios NOT found in CNEFE: 78
=== Format comparison ===
cons

In [0]:
%python


# Carregar silver
cnefe_df = spark.table("workspace.default.`29_ba`").select(
    F.concat_ws(" ", 
                F.col("NOM_TIPO_SEGLOGR"), 
                F.col("NOM_TITULO_SEGLOGR"), 
                F.col("NOM_SEGLOGR")).alias("chave_logradouro"),
    F.col("CEP").alias("chave_cep"),
    F.col("DSC_LOCALIDADE").alias("chave_bairro"),
    F.col("NUM_ENDERECO").cast("string").alias("chave_numero"),
    F.col("COD_MUNICIPIO").alias("chave_municipio"),
    F.col("COD_SETOR").alias("setor_censitario_encontrado")
).distinct()

In [0]:
%python
cnefe_df

DataFrame[chave_logradouro: string, chave_cep: bigint, chave_bairro: string, chave_numero: string, chave_municipio: bigint, setor_censitario_encontrado: string]

In [0]:
%python

# realiza fuzzy string matching
matched_df = consulta_df.alias("a").join(
    cnefe_df.alias("b"),
    # Join on municipio: consulta uses 6-digit IBGE, CNEFE uses 7-digit (drop check digit)
    F.col("a.consulta_municipio") == (F.col("b.chave_municipio") / 10).cast("long"),
    "inner"
).select(
    F.col("a.consulta_logradouro"),
    F.col("a.consulta_numero"),
    F.col("a.consulta_bairro"),
    F.col("a.consulta_cep"),
    F.col("a.consulta_complemento"),
    F.col("a.consulta_municipio"),
    F.col("b.setor_censitario_encontrado"),
    F.col("b.chave_logradouro"),
    F.col("b.chave_numero"),
    F.col("b.chave_cep"),
    F.col("b.chave_bairro"),
    # distância de levesthein - logradouro
    F.levenshtein(
        F.coalesce(F.col("a.consulta_logradouro"), F.lit("")),
        F.coalesce(F.col("b.chave_logradouro"), F.lit(""))
    ).alias("street_distance"),
    # distância de levenshtein - bairro
    F.levenshtein(
        F.coalesce(F.col("a.consulta_bairro"), F.lit("")),
        F.coalesce(F.col("b.chave_bairro"), F.lit(""))
    ).alias("bairro_distance"),
    # calcula o score
    (
        F.when(F.col("a.consulta_cep") == F.col("b.chave_cep"), 100).otherwise(0) +
        (50 - F.levenshtein(
            F.coalesce(F.col("a.consulta_logradouro"), F.lit("")),
            F.coalesce(F.col("b.chave_logradouro"), F.lit(""))
        )) +
        (30 - F.levenshtein(
            F.coalesce(F.col("a.consulta_bairro"), F.lit("")),
            F.coalesce(F.col("b.chave_bairro"), F.lit(""))
        )) +
        F.when(F.col("a.consulta_numero") == F.col("b.chave_numero"), 20).otherwise(0)
    ).alias("match_score")
).filter(
    (F.col("street_distance") <= 50) &
    (F.col("bairro_distance") <= 20)
)

print(f"\nPossíveis candidatos: {matched_df.count()}")

# Obter setor mais promissor
window_spec = Window.partitionBy(
    "consulta_logradouro", "consulta_numero", "consulta_cep"
).orderBy(F.col("match_score").desc())

best_matches = matched_df.withColumn(
    "rank", F.row_number().over(window_spec)
).filter(F.col("rank") == 1).drop("rank").orderBy(F.col("match_score").desc())

print(f"Setores encontrados: {best_matches.count()}")

display(best_matches)


Possíveis candidatos: 8396951
Setores encontrados: 107


consulta_logradouro,consulta_numero,consulta_bairro,consulta_cep,consulta_complemento,consulta_municipio,setor_censitario_encontrado,chave_logradouro,chave_numero,chave_cep,chave_bairro,street_distance,bairro_distance,match_score
RUA NOVA BRASILEIA,26,CACHA PREGOD,44470000,null,293320,293320810000001P,RUA NOVA BRASILIA,26,44470000,CACHA PREGOS,1,1,198
RUA C 14,16,PARAISO,44900000,null,291460,291460405000013P,RUA 2,16,44900000,PARAISO,4,0,196
RUA PRESIDENTE VARGAS,10,null,41611685,T,292740,292740805090106P,RUA PRESIDENTE VARGAS,10,41611685,ITAPUA,0,6,194
RUA MANOEL ALVES SAMPAIO,42,null,48730000,null,290840,290840805000002P,RUA MANUEL ALVES SAMPAIO,42,48730000,CENTRO,1,6,193
RUA ARCO VERDE,661,CLERISTON ANDRADE,45920000,SEDE,292300,292300105000026P,RUA CARDEAL ARCO VERDE,661,45920000,CLERISTON ANDRADE,8,0,192
RUA ILHEUS,17,MORADA DO PLANALTO,45190000,null,292500,292500605000026P,RUA BOA NOVA,17,45190000,MORADA DO PLANALTO,8,0,192
RUA MANGUEIRA,29,ALTO,44270000,null,293170,293170705000004P,RUA DO TERREIRO,29,44270000,ALTO,8,0,192
RUA SAO JORGE,216,OUTEIRO,45690000,null,293250,293250725000002P,RUA NOVA,216,45690000,OUTEIRO,8,0,192
TRAVESSA BURANHEN,38,BARROLANDIA,45800000,null,290340,290340915000010P,TRAVESSA JOSE ALVES,38,45800000,BARROLANDIA,8,0,192
RUA SANTO INACIO,3,SANTO INACIO,40750670,null,292740,292740805130011P,RUA SANTO INACIO,3,40750670,ALTO DE COUTOS,0,9,191


In [0]:
%python
import requests
import zipfile
import os
from pathlib import Path

# URL for Bahia census sectors shapefile
url = "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/setores/shp/UF/BA_setores_CD2022.zip"

print("📥 Downloading Bahia Census Sectors shapefile from IBGE...")
print(f"URL: {url}\n")

# Create a volume path for storing the shapefile
# First, let's create the volume if it doesn't exist
try:
    spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.geo_data")
    print("✓ Volume 'workspace.default.geo_data' ready")
except Exception as e:
    print(f"Note: {e}")

volume_path = "/Volumes/workspace/default/geo_data"
zip_file_path = f"{volume_path}/BA_setores_CD2022.zip"
extract_path = f"{volume_path}/BA_setores_2022"

# Create extraction directory
os.makedirs(extract_path, exist_ok=True)

# Download the file
try:
    print("⏳ Downloading... (this may take a minute)")
    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()
    
    # Write to volume
    with open(zip_file_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    
    file_size_mb = os.path.getsize(zip_file_path) / (1024 * 1024)
    print(f"✓ Downloaded: {file_size_mb:.2f} MB")
    
    # Extract the zip file
    print(f"\n📦 Extracting to {extract_path}...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    
    # List extracted files
    extracted_files = os.listdir(extract_path)
    print(f"\n✓ Extracted {len(extracted_files)} files:")
    for file in sorted(extracted_files):
        file_path = os.path.join(extract_path, file)
        size_kb = os.path.getsize(file_path) / 1024
        print(f"  - {file} ({size_kb:.1f} KB)")
    
    # Find the main shapefile
    shp_files = [f for f in extracted_files if f.endswith('.shp')]
    if shp_files:
        shapefile_path = f"{extract_path}/{shp_files[0]}"
        print(f"\n🎯 Main shapefile: {shapefile_path}")
        print(f"\n✅ SUCCESS! Shapefile ready for mapping.")
        print(f"\nUse this path in the next cell:")
        print(f"   shapefile_path = '{shapefile_path}'")
    else:
        print("\n⚠️  No .shp file found in the archive")
    
    # Clean up zip file to save space
    os.remove(zip_file_path)
    print(f"\n🗑️  Cleaned up zip file")
    
except requests.exceptions.RequestException as e:
    print(f"\n❌ Download failed: {e}")
    print("\nTroubleshooting:")
    print("- Check your internet connection")
    print("- Verify the IBGE server is accessible")
    print("- Try downloading manually and uploading to Databricks")
except Exception as e:
    print(f"\n❌ Error: {e}")
    print(f"\nDetails: {type(e).__name__}")

📥 Downloading Bahia Census Sectors shapefile from IBGE...
URL: https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/setores/shp/UF/BA_setores_CD2022.zip

✓ Volume 'workspace.default.geo_data' ready
⏳ Downloading... (this may take a minute)
✓ Downloaded: 59.03 MB

📦 Extracting to /Volumes/workspace/default/geo_data/BA_setores_2022...

✓ Extracted 5 files:
  - BA_setores_CD2022.cpg (0.0 KB)
  - BA_setores_CD2022.dbf (63877.2 KB)
  - BA_setores_CD2022.prj (0.1 KB)
  - BA_setores_CD2022.shp (96664.8 KB)
  - BA_setores_CD2022.shx (242.2 KB)

🎯 Main shapefile: /Volumes/workspace/default/geo_data/BA_setores_2022/BA_setores_CD2022.shp

✅ SUCCESS! Shapefile ready for mapping.

Use this path in the next cell:
   shapefile_path = '/Volumes/workspace/default/geo_data/BA_setores_2022/BA_setores_CD2022.shp'

🗑️  Cleaned up zip file


In [0]:
%python
# Install required geospatial libraries
import sys
try:
    import geopandas as gpd
    print("✓ geopandas already installed")
except ImportError:
    print("Installing geopandas...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "geopandas", "-q"])
    import geopandas as gpd

try:
    import folium
    print("✓ folium already installed")
except ImportError:
    print("Installing folium...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "folium", "-q"])
    import folium

import pandas as pd
from pyspark.sql import functions as F

print("📍 Step 1: Upload your Bahia Census Sector shapefile")
print("   Options:")
print("   a) Upload to a Unity Catalog Volume: /Volumes/<catalog>/<schema>/<volume>/")
print("   b) Upload to DBFS: /dbfs/FileStore/")
print("\n   Once uploaded, uncomment and set the 'shapefile_path' variable above.")
print("\n" + "="*60)

matched_pd = best_matches.select(
    "setor_censitario_encontrado",
    "consulta_logradouro",
    "consulta_numero",
    "consulta_cep",
    "match_score"
).toPandas()

print(f"\n📊 Matched addresses ready for visualization: {len(matched_pd)} records")
print(f"   Unique census sectors found: {matched_pd['setor_censitario_encontrado'].nunique()}")

print("\nTop 10 Census Sectors:")
print(matched_pd['setor_censitario_encontrado'].value_counts().head(10))

print("\n" + "="*60)
print("\n🗺️ Next Steps:")
print("1. Upload your shapefile (shp, gpkg, or kml) to a volume or DBFS")
print("2. Uncomment and set the shapefile_path variable above")
print("3. Run the next cell to create the map")

✓ geopandas already installed
✓ folium already installed
📍 Step 1: Upload your Bahia Census Sector shapefile
   Options:
   a) Upload to a Unity Catalog Volume: /Volumes/<catalog>/<schema>/<volume>/
   b) Upload to DBFS: /dbfs/FileStore/

   Once uploaded, uncomment and set the 'shapefile_path' variable above.


📊 Matched addresses ready for visualization: 72 records
   Unique census sectors found: 72

Top 10 Census Sectors:
setor_censitario_encontrado
292880205000098P    1
293015405000025P    1
291910805000004P    1
290135305000009P    1
292300110000005P    1
291570005000015P    1
291610405000062P    1
293200205000010P    1
290200505000022P    1
292145005000001P    1
Name: count, dtype: int64


🗺️ Next Steps:
1. Upload your shapefile (shp, gpkg, or kml) to a volume or DBFS
2. Uncomment and set the shapefile_path variable above
3. Run the next cell to create the map


In [0]:
%python
# STEP 2: Load shapefile and create the map
# Update this path after uploading your shapefile
shapefile_path = "/Volumes/workspace/default/geo_data/BA_setores_2022/BA_setores_CD2022.shp"  # <-- UPDATE THIS PATH

try:
    # Load the Bahia census sectors shapefile
    print("Loading shapefile...")
    gdf = gpd.read_file(shapefile_path)
    
    # The shapefile should have a column with the census sector code
    # Common column names: 'CD_SETOR', 'COD_SETOR', 'SETOR', etc.
    # Update the column name below if different:
    sector_column = 'CD_SETOR'  # <-- UPDATE THIS if your shapefile uses a different column name
    
    print(f"Shapefile loaded: {len(gdf)} sectors")
    print(f"Columns: {gdf.columns.tolist()}")
    print(f"\nCRS: {gdf.crs}")
    
    # Ensure CRS is WGS84 for web mapping
    if gdf.crs != 'EPSG:4326':
        gdf = gdf.to_crs('EPSG:4326')
    
    # Join with matched data
    # IMPORTANT: Strip the 'P' suffix from CNEFE sector codes to match shapefile format
    matched_pd['setor_code_clean'] = matched_pd['setor_censitario_encontrado'].astype(str).str.rstrip('P')
    gdf[sector_column] = gdf[sector_column].astype(str)
    
    # Aggregate matches per sector with address list
    sector_counts = matched_pd.groupby('setor_code_clean').agg({
        'consulta_logradouro': lambda x: list(x),
        'match_score': 'mean'
    }).reset_index()
    sector_counts['matched_count'] = sector_counts['consulta_logradouro'].apply(len)
    sector_counts['addresses'] = sector_counts['consulta_logradouro']
    sector_counts = sector_counts[['setor_code_clean', 'matched_count', 'match_score', 'addresses']]
    sector_counts.columns = ['setor_code', 'matched_count', 'avg_match_score', 'addresses']
    
    # Merge with geometry
    map_data = gdf.merge(
        sector_counts,
        left_on=sector_column,
        right_on='setor_code',
        how='left'
    )
    map_data['matched_count'] = map_data['matched_count'].fillna(0).astype(int)
    
    matched_sectors_count = (map_data['matched_count'] > 0).sum()
    print(f"\nSectors with matches: {matched_sectors_count}")
    
    # Simplify geometries to reduce HTML payload size
    print("Simplifying geometries for web rendering...")
    map_data['geometry'] = map_data['geometry'].simplify(tolerance=0.005, preserve_topology=True)
    
    # Pre-compute matched_count as integer so style_function reads from properties directly
    map_data['matched_count_int'] = map_data['matched_count'].astype(int)
    
    matched_sectors = map_data[map_data['matched_count'] > 0].copy()
    
    if len(matched_sectors) == 0:
        print("\n⚠️ Warning: No sectors matched after joining!")
        print(f"Sample sector codes from matches: {sector_counts['setor_code'].head(3).tolist()}")
        print(f"Sample sector codes from shapefile: {gdf[sector_column].head(3).tolist()}")
    
    # Create the map - center on Bahia to show all sectors
    all_bounds = map_data.total_bounds
    center_lat = (all_bounds[1] + all_bounds[3]) / 2
    center_lon = (all_bounds[0] + all_bounds[2]) / 2
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=7,
        tiles='OpenStreetMap'
    )
    
    # --- Layer 1: Municipality boundaries as gray background (dissolve 30k sectors -> ~417 mun) ---
    print("Dissolving sectors by municipality for background layer...")
    mun_boundaries = map_data.dissolve(by='NM_MUN')
    mun_boundaries = mun_boundaries.reset_index()
    mun_boundaries['geometry'] = mun_boundaries['geometry'].simplify(tolerance=0.005, preserve_topology=True)
    
    folium.GeoJson(
        mun_boundaries[['NM_MUN', 'geometry']],
        style_function=lambda x: {
            'fillColor': 'lightgray',
            'color': 'gray',
            'weight': 0.5,
            'fillOpacity': 0.15
        },
        highlight_function=lambda x: {
            'fillColor': 'yellow',
            'color': 'black',
            'weight': 1.5,
            'fillOpacity': 0.4
        },
        tooltip=folium.features.GeoJsonTooltip(
            fields=['NM_MUN'],
            aliases=['Município:'],
            localize=True
        ),
        name='Municipalities (background)'
    ).add_to(m)
    
    # --- Layer 2: Only matched sectors in GREEN (full detail, only 69 polygons) ---
    if len(matched_sectors) > 0:
        folium.GeoJson(
            matched_sectors[[sector_column, 'NM_MUN', 'NM_BAIRRO', 'matched_count_int', 'geometry']],
            style_function=lambda x: {
                'fillColor': 'green',
                'color': 'darkgreen',
                'weight': 1.5,
                'fillOpacity': 0.6
            },
            highlight_function=lambda x: {
                'fillColor': 'lime',
                'color': 'black',
                'weight': 2,
                'fillOpacity': 0.8
            },
            tooltip=folium.features.GeoJsonTooltip(
                fields=[sector_column, 'NM_MUN', 'NM_BAIRRO', 'matched_count_int'],
                aliases=['Setor:', 'Município:', 'Bairro:', 'Matches:'],
                localize=True
            ),
            name='Matched Sectors (GREEN)'
        ).add_to(m)
        
        # Add markers for matched sectors
        for idx, row in matched_sectors.iterrows():
            addresses_html = '<br>'.join([f"• {addr}" for addr in row['addresses'][:10]])
            popup_content = f"""
            <div style="font-family: Arial; font-size: 12px;">
                <b>Setor Censitário:</b> {row[sector_column]}<br>
                <b>Município:</b> {row['NM_MUN']}<br>
                <b>Bairro:</b> {row.get('NM_BAIRRO', 'N/A')}<br>
                <b>Endereços Encontrados:</b> {row['matched_count']}<br>
                <b>Score Médio:</b> {row['avg_match_score']:.0f}<br>
                <br><b>Endereços:</b><br>{addresses_html}
            </div>
            """
            centroid = row['geometry'].centroid
            folium.CircleMarker(
                location=[centroid.y, centroid.x],
                radius=6,
                color='darkgreen',
                fill=True,
                fillColor='lightgreen',
                fillOpacity=0.9,
                popup=folium.Popup(popup_content, max_width=400),
                tooltip=f"{row['matched_count']} endereços"
            ).add_to(m)
    
    folium.LayerControl().add_to(m)
    
    # Save map as HTML file for download/viewing
    map_html_path = "/Volumes/workspace/default/geo_data/bahia_map.html"
    m.save(map_html_path)
    print(f"\n📁 Map saved to: {map_html_path}")
    
    # Display the map inline
    displayHTML(m._repr_html_())
    
    print("\n✅ Map created successfully!")
    print(f"   - Total sectors in shapefile: {len(gdf):,}")
    print(f"   - Municipalities (gray background): {len(mun_boundaries)}")
    print(f"   - Matched sectors (GREEN): {matched_sectors_count}")
    print(f"   - Total matched addresses: {int(map_data['matched_count'].sum())}")
    print(f"\n🗺️ The map shows:")
    print(f"   • GRAY polygons = Bahia municipality boundaries (all {len(mun_boundaries)} municipalities)")
    print(f"   • GREEN polygons = matched census sectors ({matched_sectors_count} sectors)")
    print(f"   • Green markers = matched sector centroids (click for address details)")
    print(f"   • Hover over any area to see info")
    
except FileNotFoundError:
    print("\n❌ Shapefile not found!")
    print("\n📁 Please upload your Bahia census sector shapefile first.")
    print("\nOptions:")
    print("1. Upload to Unity Catalog Volume:")
    print("   - Navigate to Catalog > Volumes")
    print("   - Create or select a volume")
    print("   - Upload your .shp, .shx, .dbf, .prj files")
    print("\n2. Upload to DBFS:")
    print("   - Use dbutils.fs.put() or upload via Data > Add Data")
    print("\nThen update the 'shapefile_path' variable above.")
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\nTroubleshooting:")
    print("- Verify the shapefile path is correct")
    print("- Check that all shapefile components (.shp, .shx, .dbf, .prj) are uploaded")
    print(f"- Update 'sector_column' variable if your shapefile uses a different column name")

Loading shapefile...
Shapefile loaded: 30985 sectors
Columns: ['CD_SETOR', 'SITUACAO', 'CD_SIT', 'CD_TIPO', 'AREA_KM2', 'CD_REGIAO', 'NM_REGIAO', 'CD_UF', 'NM_UF', 'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST', 'CD_BAIRRO', 'NM_BAIRRO', 'CD_NU', 'NM_NU', 'CD_FCU', 'NM_FCU', 'CD_AGLOM', 'NM_AGLOM', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI', 'CD_CONCURB', 'NM_CONCURB', 'geometry']

CRS: EPSG:4674

Sectors with matches: 69
Simplifying geometries for web rendering...
Dissolving sectors by municipality for background layer...

📁 Map saved to: /Volumes/workspace/default/geo_data/bahia_map.html


Make this Notebook Trusted to load map: File -> Trust Notebook <iframe srcdoc="<!DOCTYPE html>
<html>
<head>
 
 <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
 <script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.js"></script>
 <script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
 <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/js/bootstrap.bundle.min.js"></script>
 <script src="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.js"></script>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.3/dist/leaflet.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.2.2/dist/css/bootstrap.min.css"/>
 <link rel="stylesheet" href="https://netdna.bootstrapcdn.com/bootstrap/3.0.0/css/bootstrap-glyphicons.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@fortawesome/fontawesome-free@6.2.0/css/all.min.css"/>
 <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/Leaflet.awesome-markers/2.0.2/leaflet.awesome-markers.css"/>
 <link rel="stylesheet" href="https://cdn.jsdelivr.net/gh/python-visualization/folium/folium/templates/leaflet.awesome.rotate.min.css"/>
 
 <meta name="viewport" content="width=device-width,
 initial-scale=1.0, maximum-scale=1.0, user-scalable=no" />
 <style>
 #map_98dd106611165ec87811864dba631596 {
 position: relative;
 width: 100.0%;
 height: 100.0%;
 left: 0.0%;
 top: 0.0%;
 }
 .leaflet-container { font-size: 1rem; }
 </style>

 <style>html, body {
 width: 100%;
 height: 100%;
 margin: 0;
 padding: 0;
 }
 </style>

 <style>#map {
 position:absolute;
 top:0;
 bottom:0;
 right:0;
 left:0;
 }
 </style>

 <script>
 L_NO_TOUCH = false;
 L_DISABLE_3D = false;
 </script>

 
 
 <style>
 .foliumtooltip {
 
 }
 .foliumtooltip table{
 margin: auto;
 }
 .foliumtooltip tr{
 text-align: left;
 }
 .foliumtooltip th{
 padding: 2px; padding-right: 8px;
 }
 </style>
 
 
 <style>
 .foliumtooltip {
 
 }
 .foliumtooltip table{
 margin: auto;
 }
 .foliumtooltip tr{
 text-align: left;
 }
 .foliumtooltip th{
 padding: 2px; padding-right: 8px;
 }
 </style>
 
</head>
<body>
 
 
 <div class="folium-map" id="map_98dd106611165ec87811864dba631596" ></div>
 
</body>
<script>
 
 
 var map_98dd106611165ec87811864dba631596 = L.map(
 "map_98dd106611165ec87811864dba631596",
 {
 center: [-13.440690449999977, -41.95913064999996],
 crs: L.CRS.EPSG3857,
 ...{
 "zoom": 7,
 "zoomControl": true,
 "preferCanvas": false,
}

 }
 );

 

 
 
 var tile_layer_b49e1862ce379ccbbd04b7644b1a8208 = L.tileLayer(
 "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
 {
 "minZoom": 0,
 "maxZoom": 19,
 "maxNativeZoom": 19,
 "noWrap": false,
 "attribution": "\u0026copy; \u003ca href=\"https://www.openstreetmap.org/copyright\"\u003eOpenStreetMap\u003c/a\u003e contributors",
 "subdomains": "abc",
 "detectRetina": false,
 "tms": false,
 "opacity": 1,
}

 );
 
 
 tile_layer_b49e1862ce379ccbbd04b7644b1a8208.addTo(map_98dd106611165ec87811864dba631596);
 
 
 function geo_json_384f37bea88536f3ee00e9edeb172206_styler(feature) {
 switch(feature.id) {
 default:
 return {"color": "gray", "fillColor": "lightgray", "fillOpacity": 0.15, "weight": 0.5};
 }
 }
 function geo_json_384f37bea88536f3ee00e9edeb172206_highlighter(feature) {
 switch(feature.id) {
 default:
 return {"color": "black", "fillColor": "yellow", "fillOpacity": 0.4, "weight": 1.5};
 }
 }

 function geo_json_384f37bea88536f3ee00e9edeb172206_onEachFeature(feature, layer) {

 layer.on({
 mouseout: function(e) {
 if(typeof e.target.setStyle === "function"){
 geo_json_384f37bea88536f3ee00e9edeb172206.resetStyle(e.target);
 }
 },
 mouseover: function(e) {
 if(typeof e.target.setStyle === "function"){
 const highlightStyle = geo_json_384f37bea88536f3ee00e9edeb172206_highlighter(e.target.feature)
 e.target.setStyle(highlightStyle);
 }
 },
 });
 };
 var geo_json_384f37bea88536f3ee00e9edeb172206 = L.geoJson(null, {
 onEachFeature: geo_json_384f37bea88536f3


✅ Map created successfully!
   - Total sectors in shapefile: 30,985
   - Municipalities (gray background): 417
   - Matched sectors (GREEN): 69
   - Total matched addresses: 69

🗺️ The map shows:
   • GRAY polygons = Bahia municipality boundaries (all 417 municipalities)
   • GREEN polygons = matched census sectors (69 sectors)
   • Green markers = matched sector centroids (click for address details)
   • Hover over any area to see info
